<h1>Linear regression with PyTorch</h1>



![linear-regression-training-data](https://i.imgur.com/6Ujttb4.png)



<h2>Installations</h2>

In [4]:
!pip install torch numpy --quiet

In [5]:
import torch 
import numpy as np

<h2>Training Data</h2>

<h3>Inputs: Temperature, Rainfall, Humidity</h3>

In [8]:
inputs = np.array([[73, 67, 43],
                   [91, 88, 64],
                   [87, 134, 58],
                   [102, 43, 37],
                   [69, 96, 70]], dtype="float32")

<h3>Targets: Apples, Oranges</h3>

In [10]:
targets = np.array([[56, 70],
                    [81, 101],
                    [119, 133],
                    [22, 37],
                    [103, 119]], dtype="float32")

<h3>Numpy to Tensors</h3>

In [12]:
inputs = torch.from_numpy(inputs)
targets = torch.from_numpy(targets)
print(inputs)
print(targets)

tensor([[ 73.,  67.,  43.],
        [ 91.,  88.,  64.],
        [ 87., 134.,  58.],
        [102.,  43.,  37.],
        [ 69.,  96.,  70.]])
tensor([[ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.]])


<h3>Weights and Biases</h3>

In [14]:
w = torch.randn(2, 3, requires_grad=True)
b = torch.randn(2, requires_grad=True)
print(w)
print(b)

tensor([[-0.9357,  2.0531, -0.2780],
        [-0.5725, -1.6275, -0.3247]], requires_grad=True)
tensor([-0.7872, -0.3387], requires_grad=True)


In [15]:
def model(x):
    return x @ w.t() + b

<h3>Predictions</h3>

In [17]:
preds = model(inputs)
print(preds)

tensor([[  56.5105, -165.1363],
        [  76.9451, -216.4380],
        [ 176.7992, -287.0640],
        [ -18.2318, -140.7306],
        [ 112.2876, -218.8115]], grad_fn=<AddBackward0>)


In [18]:
print(targets)

tensor([[ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.]])


<h2>Loss Functions</h2>

<h3>MSE Loss</h3>

In [21]:
def mse(t1, t2):
    diff = t1 - t2
    return torch.sum(diff*diff) / diff.numel()

In [22]:
losses = mse(preds, targets)
losses

tensor(48327.6758, grad_fn=<DivBackward0>)

<h2>Adjust Weights and Biases using Gradient Decent</h2>

In [24]:
pred = model(inputs)
preds

tensor([[  56.5105, -165.1363],
        [  76.9451, -216.4380],
        [ 176.7992, -287.0640],
        [ -18.2318, -140.7306],
        [ 112.2876, -218.8115]], grad_fn=<AddBackward0>)

In [25]:
loss =  mse(preds, targets)
loss

tensor(48327.6758, grad_fn=<DivBackward0>)

In [26]:
loss.backward()
print(w.grad)
print(b.grad)

tensor([[   246.8006,   1316.8212,    455.2693],
        [-24806.9785, -28009.9141, -17002.6875]])
tensor([   4.6621, -297.6361])


In [27]:
with torch.no_grad():
    w -= w.grad * 1e-5
    b -= b.grad * 1e-5
    w.grad.zero_()
    b.grad.zero_()

In [28]:
print(w)
print(b)

tensor([[-0.9382,  2.0399, -0.2826],
        [-0.3244, -1.3474, -0.1547]], requires_grad=True)
tensor([-0.7873, -0.3357], requires_grad=True)


In [29]:
preds = model(inputs)
loss = mse(preds, targets)
loss

tensor(32929.6016, grad_fn=<DivBackward0>)

<h2>Train for multiple epoch</h2>

In [31]:
for i in range(100):
    preds = model(inputs)
    loss = mse(preds, targets)
    loss.backward()
    with torch.no_grad():
        w -= w.grad * 1e-5
        b -= b.grad * 1e-5

In [32]:
preds = model(inputs)
loss = mse(preds, targets)
loss

tensor(6222.3008, grad_fn=<DivBackward0>)

In [33]:
preds

tensor([[ 59.9924, -20.6890],
        [ 94.8800, -16.3312],
        [101.5778,  -2.5248],
        [ 26.4454, -62.2718],
        [119.5081,  13.1433]], grad_fn=<AddBackward0>)

In [34]:
targets

tensor([[ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.]])

<h2>Linear Regression using PyTorch built-ins</h2>

In [58]:
import torch.nn as nn

In [60]:
# Input (temp, rainfall, humidity)
inputs = np.array([[73, 67, 43], [91, 88, 64], [87, 134, 58], 
                   [102, 43, 37], [69, 96, 70], [73, 67, 43], 
                   [91, 88, 64], [87, 134, 58], [102, 43, 37], 
                   [69, 96, 70], [73, 67, 43], [91, 88, 64], 
                   [87, 134, 58], [102, 43, 37], [69, 96, 70]], 
                  dtype='float32')

# Targets (apples, oranges)
targets = np.array([[56, 70], [81, 101], [119, 133], 
                    [22, 37], [103, 119], [56, 70], 
                    [81, 101], [119, 133], [22, 37], 
                    [103, 119], [56, 70], [81, 101], 
                    [119, 133], [22, 37], [103, 119]], 
                   dtype='float32')

inputs = torch.from_numpy(inputs)
targets = torch.from_numpy(targets)

<h3>Datasets and DataLoader</h3>

In [64]:
from torch.utils.data import TensorDataset

In [68]:
train_ds = TensorDataset(inputs, targets)
train_ds[0:3]

(tensor([[ 73.,  67.,  43.],
         [ 91.,  88.,  64.],
         [ 87., 134.,  58.]]),
 tensor([[ 56.,  70.],
         [ 81., 101.],
         [119., 133.]]))

In [70]:
from torch.utils.data import DataLoader

In [72]:
batch_size = 5
train_dl = DataLoader(train_ds, batch_size, shuffle=True)

In [74]:
for xb, yb in train_dl:
    print(xb)
    print(yb)
    break

tensor([[ 69.,  96.,  70.],
        [ 87., 134.,  58.],
        [ 69.,  96.,  70.],
        [ 91.,  88.,  64.],
        [ 69.,  96.,  70.]])
tensor([[103., 119.],
        [119., 133.],
        [103., 119.],
        [ 81., 101.],
        [103., 119.]])


<h3>nn.Linear</h3>

In [78]:
model = nn.Linear(3, 2)
print(model.weight)
print(model.bias)

Parameter containing:
tensor([[-0.4380, -0.4803, -0.1178],
        [-0.0361,  0.1327,  0.3012]], requires_grad=True)
Parameter containing:
tensor([0.3301, 0.3274], requires_grad=True)


In [80]:
list(model.parameters())

[Parameter containing:
 tensor([[-0.4380, -0.4803, -0.1178],
         [-0.0361,  0.1327,  0.3012]], requires_grad=True),
 Parameter containing:
 tensor([0.3301, 0.3274], requires_grad=True)]

In [82]:
pred = model(inputs)
preds

tensor([[ 59.9924, -20.6890],
        [ 94.8800, -16.3312],
        [101.5778,  -2.5248],
        [ 26.4454, -62.2718],
        [119.5081,  13.1433]], grad_fn=<AddBackward0>)

<h3>Loss Function</h3>

In [86]:
import torch.nn.functional as F

In [92]:
loss_fn = F.mse_loss

In [96]:
loss = loss_fn(model(inputs), targets)
print(loss)

tensor(16615.2051, grad_fn=<MseLossBackward0>)


<h3>Optimizer</h3>

In [99]:
opt = torch.optim.SGD(model.parameters(), lr=1e-5)

<h3>Train the model</h3>

In [106]:
def fit(num_epochs, model, loss_fn, opt):
    for epoch in range(num_epochs):
        
        #Train with batches
        for xb, yb in train_dl:

            #1. Generate predictions
            pred = model(xb)

            #2. Claculate Loss
            loss = loss_fn(pred, yb)

            #3. Compute Gradient
            loss.backward()

            #4. Update parameters using gradient
            opt.step()

            #5. Reset Gradient to zero
            opt.zero_grad()

        #Print the progress
        if (epoch+1) % 10 == 0:
            print('Epoch [{}/{}], Loss: {:.4f}'.format(epoch+1, num_epochs, loss.item()))

In [108]:
fit(100, model, loss_fn, opt)

Epoch [10/100], Loss: 109.6108
Epoch [20/100], Loss: 312.6672
Epoch [30/100], Loss: 86.2328
Epoch [40/100], Loss: 62.2737
Epoch [50/100], Loss: 47.2496
Epoch [60/100], Loss: 21.2882
Epoch [70/100], Loss: 49.3087
Epoch [80/100], Loss: 12.5133
Epoch [90/100], Loss: 35.2807
Epoch [100/100], Loss: 8.7086


In [114]:
preds = model(inputs)
preds

tensor([[ 58.3224,  71.2817],
        [ 81.5824,  99.1964],
        [117.7608, 134.1767],
        [ 27.7318,  42.5878],
        [ 97.0549, 113.4282],
        [ 58.3224,  71.2817],
        [ 81.5824,  99.1964],
        [117.7608, 134.1767],
        [ 27.7318,  42.5878],
        [ 97.0549, 113.4282],
        [ 58.3224,  71.2817],
        [ 81.5824,  99.1964],
        [117.7608, 134.1767],
        [ 27.7318,  42.5878],
        [ 97.0549, 113.4282]], grad_fn=<AddmmBackward0>)

In [116]:
targets

tensor([[ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.],
        [ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.],
        [ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.]])